In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate, StratifiedKFold, cross_val_predict
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    make_scorer
)

**Load data**

In [ ]:
X_train = pd.read_csv("X_train_final.csv")
X_test = pd.read_csv("X_test_final.csv")

y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

In [ ]:
X_train.shape

(273, 99)

In [4]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LGBMClassifier(
    n_estimators=300,
    learning_rate=0.02,
    class_weight="balanced",
    num_leaves=15,
    min_data_in_leaf=5,
    min_gain_to_split=0,
    random_state=42
    ))
])

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
scoring = {
    "macro_f1": make_scorer(f1_score, average="macro")
}

In [7]:
cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

In [8]:
macro_f1_scores = cv_results["test_macro_f1"]

print("CV Macro F1 (default threshold): %.3f ± %.3f" %
      (macro_f1_scores.mean(), macro_f1_scores.std()))

CV Macro F1 (default threshold): 0.827 ± 0.041


In [9]:
y_prob_train = cross_val_predict(
    pipeline,
    X_train,
    y_train, # Ensure y_train is a 1D array
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:,1]

In [10]:
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = []

for t in thresholds:
    preds = (y_prob_train >= t).astype(int)
    f1_scores.append(f1_score(y_train, preds, average="macro"))

best_threshold = thresholds[np.argmax(f1_scores)]

print("Best threshold:", round(best_threshold,3))
print("Macro F1 after threshold tuning:", round(max(f1_scores),3))

Best threshold: 0.41
Macro F1 after threshold tuning: 0.828


In [11]:
pipeline.fit(X_train, y_train)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_gain_to_split is set=0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_gain_to_split is set=0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0
[LightGBM] [Info] Number of positive: 82, number of negative: 191
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000502 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8135
[LightGBM] [Info] Number of data points in the train set: 273, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LGBMClassifier(class_weight='balanced', learning_rate=0.02,
                                min_data_in_leaf=5, min_gain_to_split=0,
                                n_estimators=300, num_leaves=15,
                                random_state=42))])

In [12]:
y_prob = pipeline.predict_proba(X_test)[:,1]

[LightGBM] [Warning] min_gain_to_split is set=0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
y_pred = (y_prob >= best_threshold).astype(int)

In [14]:
macro_f1 = f1_score(y_test, y_pred, average="macro")
balanced_acc = balanced_accuracy_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("TEST SET PERFORMANCE")
print("--------------------")
print(f"Macro F1: {macro_f1:.2f}")
print(f"Balanced Accuracy: {balanced_acc:.2f}")
print(f"MCC: {mcc:.2f}")
print(f"ROC-AUC: {roc_auc:.2f}")
print(f"PR-AUC: {pr_auc:.2f}")

TEST SET PERFORMANCE
--------------------
Macro F1: 0.79
Balanced Accuracy: 0.79
MCC: 0.59
ROC-AUC: 0.87
PR-AUC: 0.71
